# Triagegeist: flagging undertriage at the triage desk

In the emergency department a triage nurse assigns an acuity level (ESI 1 to 5) in about a minute,
usually from a one-line chief complaint and a set of vital signs. The most dangerous mistake is
undertriage: a patient who needs care within minutes (ESI 1 or 2) is given a lower priority and waits.

This notebook builds a flag for those patients. It reads the chief complaint and the intake data
that exist at the moment of triage, sets its threshold by how many high-acuity patients we are
willing to miss, adds a rule-based safety net on dangerous vital signs, and checks whether the
misses fall more heavily on some patient groups than others.

The competition data is synthetic. Section 2 tests how far it can be trusted, and section 21
re-fits the main text model on real emergency department visits from the CDC's NHAMCS survey.

This was written for the Triagegeist hackathon on Kaggle, which was cancelled before any entry
was judged. The README in the GitHub repository has the details.

## 1. Problem definition

- Target: `triage_acuity`, the ESI level from 1 to 5, where 1 is the most urgent.
- Main task: the binary label `high_risk = triage_acuity <= 2`. Can we catch the patients who must not wait?
- The two kinds of error do not cost the same. A missed high-risk patient can deteriorate in the
  waiting room. A false alarm costs a nurse a second look. We report sensitivity on the high-risk
  class first, then precision and the share of patients flagged.

In [ ]:
import os, sys, re, glob, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False

ASSETS = "assets" if os.path.isdir("assets") else "."
print(f"python {sys.version.split()[0]}, numpy {np.__version__}, pandas {pd.__version__}, "
      f"scikit-learn {sklearn.__version__}, torch {torch.__version__ if HAS_TORCH else 'none'}")

In [ ]:
COMP = next((p for p in ["/kaggle/input/competitions/triagegeist", "/kaggle/input/triagegeist", "data"]
             if os.path.isdir(p)), "data")
PATIENT_ID = "patient_id"
TEXT_COL = "chief_complaint_raw"
TARGET = "triage_acuity"

train = pd.read_csv(f"{COMP}/train.csv")
test = pd.read_csv(f"{COMP}/test.csv")
chief = pd.read_csv(f"{COMP}/chief_complaints.csv")
patient_hx = pd.read_csv(f"{COMP}/patient_history.csv")

train = train.merge(chief[[PATIENT_ID, TEXT_COL]], on=PATIENT_ID, how="left")
test = test.merge(chief[[PATIENT_ID, TEXT_COL]], on=PATIENT_ID, how="left")
train = train.merge(patient_hx, on=PATIENT_ID, how="left")
test = test.merge(patient_hx, on=PATIENT_ID, how="left")

overlap = set(train[PATIENT_ID].dropna().unique()) & set(test[PATIENT_ID].dropna().unique())
assert len(overlap) == 0, f"train/test patient_id overlap: {len(overlap)} rows"
missing_cc = train[TEXT_COL].isna().sum() + test[TEXT_COL].isna().sum()
assert missing_cc == 0, f"missing chief complaints: {missing_cc}"
print("train", train.shape, "test", test.shape)

In [ ]:
train["high_risk"] = (train[TARGET] <= 2).astype(int)

def present(cols):
    return [c for c in cols if c in train.columns]

LEAK_OUTCOME = present(["disposition", "ed_los_hours"])
ENGINEERED = present(["news2_score", "shock_index"])
FAIRNESS_AXES = present(["sex", "age_group", "language", "insurance_type"])

for df_ in (train, test):
    if "pain_score" in df_.columns:
        df_.loc[df_["pain_score"] == -1, "pain_score"] = np.nan

blank_cc = int(train[TEXT_COL].fillna("").str.strip().eq("").sum())
print(f"duplicate ids: {int(train[PATIENT_ID].duplicated().sum())}, "
      f"missing target: {int(train[TARGET].isna().sum())}, blank complaints: {blank_cc}")

## 2. Can the synthetic data be trusted?

Two problems would make any result on simulated data meaningless:

1. Templated text. If the chief complaints come from a small fixed list, a text model scores well
   by memorising the list, and the result says nothing about real triage notes.
2. Leakage by construction. If an engineered score such as `news2_score` is close to a direct
   function of the label, a model that uses it is reading the answer back.

We test both before any modelling.

In [ ]:
cc = train[TEXT_COL].fillna("").str.strip()
n_unique = cc.nunique()
print(f"{n_unique} unique complaints in {len(cc)} rows ({n_unique / len(cc):.1%}), "
      f"mean length {cc.str.len().mean():.1f} chars, max {cc.str.len().max()}")
print(cc.sample(10, random_state=SEED).tolist())

In [ ]:
from sklearn.metrics import roc_auc_score

ENGINEERED_LEAKY, leak_auc = [], {}
for col in ENGINEERED:
    s = train[[col, "high_risk"]].dropna()
    if len(s) == 0 or s["high_risk"].nunique() < 2:
        continue
    auc = roc_auc_score(s["high_risk"], s[col])
    leak_auc[col] = max(auc, 1 - auc)
    if leak_auc[col] > 0.90:
        ENGINEERED_LEAKY.append(col)
print("single-feature AUC:", {k: round(float(v), 3) for k, v in leak_auc.items()}, "leaky:", ENGINEERED_LEAKY)

plt.figure(figsize=(5, 3))
plt.barh(list(leak_auc), list(leak_auc.values()),
         color=["#E45756" if v > 0.9 else "#4C78A8" for v in leak_auc.values()])
plt.axvline(0.9, ls="--", color="gray")
plt.xlim(0.5, 1.02)
plt.xlabel("single-feature AUC vs high_risk")
plt.title("Leakage check: engineered scores")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "leakage_auc.png"), dpi=90)
plt.show()

## 3. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
train[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#4C78A8")
ax[0].set_title("ESI acuity distribution")
ax[0].set_xlabel("ESI level")
ax[0].set_ylabel("count")
train["high_risk"].value_counts().sort_index().plot(kind="bar", ax=ax[1], color="#E45756")
ax[1].set_title("Binary high-risk (ESI<=2)")
ax[1].set_xticklabels(["low (3-5)", "high (1-2)"], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "eda_label_dist.png"), dpi=90)
plt.show()
print(f"high-risk prevalence: {train['high_risk'].mean():.1%}")

In [ ]:
miss_cols = [c for c in ["systolic_bp", "diastolic_bp", "respiratory_rate", "temperature_c", "spo2",
                         "pain_score"] if c in train]
by_ac = train.groupby(TARGET)[miss_cols].apply(lambda d: d.isna().mean())
print((by_ac * 100).round(1).to_string())

plt.figure(figsize=(7, 3))
plt.imshow(by_ac.values, aspect="auto", cmap="Reds")
plt.colorbar(label="missing fraction")
plt.yticks(range(len(by_ac.index)), [f"ESI {i}" for i in by_ac.index])
plt.xticks(range(len(by_ac.columns)), by_ac.columns, rotation=30, ha="right")
plt.title("Missingness by ESI level")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "missingness.png"), dpi=90)
plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(min_df=3, token_pattern=r"[A-Za-z/]+")
X = cv.fit_transform(train[TEXT_COL].fillna(""))
vocab = np.array(cv.get_feature_names_out())
hi = np.asarray(X[train["high_risk"].values == 1].sum(0)).ravel()
lo = np.asarray(X[train["high_risk"].values == 0].sum(0)).ravel()
rate = (hi + 1) / (hi + lo + 2)
order = np.argsort(rate)

_hi, _lo = order[::-1][:12], order[:12]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].barh([vocab[i] for i in _hi][::-1], [rate[i] for i in _hi][::-1], color="#E45756")
ax[0].set_title("Top high-risk tokens")
ax[0].set_xlim(0, 1)
ax[0].set_xlabel("P(high | token)")
ax[1].barh([vocab[i] for i in _lo][::-1], [rate[i] for i in _lo][::-1], color="#4C78A8")
ax[1].set_title("Top low-risk tokens")
ax[1].set_xlim(0, 1)
ax[1].set_xlabel("P(high | token)")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "top_tokens.png"), dpi=90)
plt.show()

In [ ]:
fig, axx = plt.subplots(1, len(FAIRNESS_AXES), figsize=(4 * len(FAIRNESS_AXES), 3.2))
for _ax, _axis in zip(np.atleast_1d(axx), FAIRNESS_AXES):
    _g = train.groupby(_axis)["high_risk"].mean() * 100
    _ax.bar(_g.index.astype(str), _g.values, color="#4C78A8")
    _ax.set_title(f"High-risk % by {_axis}")
    _ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "highrisk_by_group.png"), dpi=90)
plt.show()

### Vitals, comorbidities and complaint keywords by ESI level

In [ ]:
_vit = [c for c in ["systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate",
                    "temperature_c", "spo2", "gcs_total", "pain_score"] if c in train.columns]
_mv = train.groupby(TARGET)[_vit].median()
_z = (_mv - _mv.mean()) / (_mv.std() + 1e-9)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(_z.values, aspect="auto", cmap="coolwarm")
ax[0].set_xticks(range(len(_vit)))
ax[0].set_xticklabels(_vit, rotation=40, ha="right")
ax[0].set_yticks(range(len(_mv.index)))
ax[0].set_yticklabels([f"ESI {i}" for i in _mv.index])
ax[0].set_title("Median vitals by ESI (z-scored)")
fig.colorbar(im, ax=ax[0])

_hx = [c for c in train.columns if c.startswith("hx_") and not c.startswith("hx_sys")]
if _hx:
    _cm = train.groupby(TARGET)[_hx].mean()
    im2 = ax[1].imshow(_cm.values, aspect="auto", cmap="Reds")
    ax[1].set_xticks(range(len(_hx)))
    ax[1].set_xticklabels([c[3:] for c in _hx], rotation=70, ha="right", fontsize=7)
    ax[1].set_yticks(range(len(_cm.index)))
    ax[1].set_yticklabels([f"ESI {i}" for i in _cm.index])
    ax[1].set_title("Comorbidity prevalence by ESI")
    fig.colorbar(im2, ax=ax[1])
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "clinical_heatmaps.png"), dpi=90)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
if "disposition" in train.columns:
    _d = pd.crosstab(train[TARGET], train["disposition"], normalize="index")
    _d.plot(kind="bar", stacked=True, ax=ax[0], colormap="viridis")
    ax[0].set_title("Disposition mix by ESI")
    ax[0].set_xlabel("ESI")
    ax[0].legend(fontsize=7)
KW = {"cardiac": ["chest pain", "cardiac"], "resp": ["shortness of breath", "sob", "dyspnoea", "dyspnea"],
      "neuro": ["stroke", "seizure", "unconscious", "altered"], "trauma": ["gsw", "mvc", "wound", "fracture"],
      "sepsis": ["sepsis", "fever", "infection"], "minor": ["mild", "refill", "sore throat", "rash"]}
_cctext = train[TEXT_COL].fillna("").str.lower()
_kw = pd.concat({k: _cctext.apply(lambda t, w=words: any(x in t for x in w)).groupby(train[TARGET]).mean()
                 for k, words in KW.items()}, axis=1)
im3 = ax[1].imshow(_kw.T.values, aspect="auto", cmap="Purples")
ax[1].set_yticks(range(len(_kw.columns)))
ax[1].set_yticklabels(list(_kw.columns))
ax[1].set_xticks(range(len(_kw.index)))
ax[1].set_xticklabels([f"ESI {i}" for i in _kw.index])
ax[1].set_title("Keyword-group prevalence by ESI")
fig.colorbar(im3, ax=ax[1])
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "disposition_keyword.png"), dpi=90)
plt.show()

## 4. Preprocessing

1. Split first, fit second. Every vectorizer and encoder is fit on the training split only, so
   nothing from the validation rows leaks into the models.
2. Leave short text mostly alone. Chief complaints are two to five words, so removing stopwords
   or negations throws away signal. We lowercase, expand common abbreviations (SOB becomes
   shortness of breath) and strip stray punctuation.

In [ ]:
from sklearn.model_selection import train_test_split

tr, va = train_test_split(train, test_size=0.2, stratify=train["high_risk"], random_state=SEED)
tr, va = tr.copy(), va.copy()
print(f"train {len(tr)} rows, valid {len(va)} rows, high-risk {tr['high_risk'].mean():.1%} / "
      f"{va['high_risk'].mean():.1%}")

In [ ]:
ABBREV = {
    "sob": "shortness of breath", "cp": "chest pain", "n/v": "nausea vomiting",
    "abd": "abdominal", "doe": "dyspnea on exertion", "loc": "loss of consciousness",
    "gsw": "gunshot wound", "mvc": "motor vehicle collision", "etoh": "alcohol",
    "ams": "altered mental status", "od": "overdose", "sx": "symptoms",
    "c/o": "complains of", "pt": "patient", "uti": "urinary tract infection",
    "dka": "diabetic ketoacidosis", "gi": "gastrointestinal", "sz": "seizure",
}

text_lower = train[TEXT_COL].fillna("").str.lower()
found_abbrev = {k: int(text_lower.str.contains(k, regex=False).sum()) for k in ABBREV}
found_abbrev = {k: n for k, n in found_abbrev.items() if n > 0}
if not found_abbrev:
    ABBREV = {}
print("abbreviations found:", found_abbrev)

def clean_text(t):
    t = str(t).lower().strip()
    for k, v in ABBREV.items():
        t = re.sub(r"(?<![a-z])" + re.escape(k) + r"(?![a-z])", v, t)
    t = re.sub(r"[^a-z0-9/\-\s]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

for df in (tr, va, train, test):
    df["cc_clean"] = df[TEXT_COL].fillna("").map(clean_text)

In [ ]:
def add_interactions(df):
    g = lambda c: df[c] if c in df.columns else pd.Series(np.nan, index=df.index)
    sbp, hr, spo2, gcs, rr = g("systolic_bp"), g("heart_rate"), g("spo2"), g("gcs_total"), g("respiratory_rate")
    age, nco = g("age"), g("num_comorbidities")
    altered = (df["mental_status_triage"].astype(str).str.lower().eq("altered")
               if "mental_status_triage" in df.columns else (gcs < 15))
    df["ix_hypotensive_tachycardic"] = ((sbp < 100) & (hr > 100)).fillna(False).astype(int)
    df["ix_elderly_multimorbid"] = ((age >= 65) & (nco >= 3)).fillna(False).astype(int)
    df["ix_altered_hypoxic"] = ((altered | (gcs <= 12)) & (spo2 < 92)).fillna(False).astype(int)
    df["ix_hypoxic_tachypneic"] = ((spo2 < 92) & (rr > 22)).fillna(False).astype(int)
    return ["ix_hypotensive_tachycardic", "ix_elderly_multimorbid",
            "ix_altered_hypoxic", "ix_hypoxic_tachypneic"]

for _df in (tr, va, train, test):
    IX_COLS = add_interactions(_df)
print({c: round(float(train[c].mean()), 3) for c in IX_COLS})

In [ ]:
def add_clinical_features(df):
    g = lambda c: df[c] if c in df.columns else pd.Series(np.nan, index=df.index)
    sbp, dbp, hr, rr, spo2, temp, gcs, age = (g("systolic_bp"), g("diastolic_bp"), g("heart_rate"),
        g("respiratory_rate"), g("spo2"), g("temperature_c"), g("gcs_total"), g("age"))
    altered = (df["mental_status_triage"].astype(str).str.lower().eq("altered")
               if "mental_status_triage" in df.columns else (gcs < 15))
    mapv = g("mean_arterial_pressure")
    mapv = mapv.where(mapv.notna(), (sbp + 2 * dbp) / 3)

    df["qsofa"] = ((rr >= 22).fillna(False).astype(int) + (sbp <= 100).fillna(False).astype(int)
                   + altered.fillna(False).astype(int))
    df["sirs"] = (((temp > 38) | (temp < 36)).fillna(False).astype(int)
                  + (hr > 90).fillna(False).astype(int) + (rr > 20).fillna(False).astype(int))
    df["rate_pressure_product"] = hr * sbp
    df["mod_shock_index"] = hr / mapv
    df["spo2_rr_ratio"] = spo2 / rr
    si = hr / sbp
    df["peds_shock_index"] = np.where(age < 2, si / 1.4, np.where(age < 10, si / 1.2, si))

    cci_w = {"hx_heart_failure": 1, "hx_cad": 1, "hx_copd": 1, "hx_diabetes": 1, "hx_stroke": 1,
             "hx_dementia": 1, "hx_ckd": 2, "hx_malignancy": 2, "hx_liver_disease": 3}
    df["charlson_index"] = sum(df[c].fillna(0) * w for c, w in cci_w.items() if c in df.columns) \
        if any(c in df.columns for c in cci_w) else 0
    SYS = {"hx_sys_cardio": ["hx_hypertension", "hx_heart_failure", "hx_cad", "hx_afib", "hx_pad"],
           "hx_sys_resp": ["hx_copd", "hx_asthma"],
           "hx_sys_neuro": ["hx_stroke", "hx_dementia", "hx_seizure"],
           "hx_sys_metabolic": ["hx_diabetes", "hx_ckd", "hx_obesity", "hx_thyroid", "hx_liver_disease"],
           "hx_sys_onco_immuno": ["hx_malignancy", "hx_immunocompromised", "hx_anemia"]}
    for name, cols in SYS.items():
        present_cols = [c for c in cols if c in df.columns]
        df[name] = df[present_cols].fillna(0).sum(axis=1) if present_cols else 0

    vit = [c for c in ["systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate",
                       "temperature_c", "spo2", "pain_score"] if c in df.columns]
    df["n_vitals_missing"] = df[vit].isna().sum(axis=1)

    if "arrival_hour" in df.columns:
        df["hour_sin"] = np.sin(2 * np.pi * df["arrival_hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["arrival_hour"] / 24)
    if "arrival_day" in df.columns:
        dmap = {d: i for i, d in enumerate(["Monday", "Tuesday", "Wednesday", "Thursday",
                                            "Friday", "Saturday", "Sunday"])}
        dn = df["arrival_day"].map(dmap)
        df["day_sin"] = np.sin(2 * np.pi * dn / 7)
        df["day_cos"] = np.cos(2 * np.pi * dn / 7)
    if "num_prior_ed_visits_12m" in df.columns:
        df["frequent_flyer"] = (df["num_prior_ed_visits_12m"] >= 4).astype(int)
        df["log_prior_visits"] = np.log1p(df["num_prior_ed_visits_12m"])
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

for _df in (tr, va, train, test):
    add_clinical_features(_df)

## 5. Evaluation protocol

High-risk patients are the rarer and more costly class, so accuracy and ROC-AUC are not the
lead metrics. The threshold for every model is the highest one that keeps undertriage at or
below 2% on the validation split (sensitivity of at least 98% for ESI 1-2). At that threshold
we report sensitivity, precision and the alert rate (the share of all patients flagged, which
is the extra workload for staff), along with PR-AUC, ROC-AUC and the Brier score. The same
code scores the structured, text and transformer models.

In [ ]:
from sklearn.metrics import (average_precision_score, brier_score_loss, precision_recall_curve,
                             recall_score, precision_score, f1_score, confusion_matrix)
from sklearn.calibration import calibration_curve

RESULTS = {}
MAX_UNDERTRIAGE = 0.02

def pick_threshold(y, p, target_sensitivity=1 - MAX_UNDERTRIAGE):
    y, p = np.asarray(y), np.asarray(p)
    if np.unique(y).size < 2 or len(p) == 0:
        return float(np.min(p)) if len(p) else 0.5
    prec, rec, thr = precision_recall_curve(y, p)
    if len(thr) == 0:
        return float(np.min(p))
    ok = rec[:-1] >= target_sensitivity
    if ok.any():
        idx = np.where(ok)[0]
        return float(thr[idx[np.argmax(thr[idx])]])
    print(f"pick_threshold: sensitivity {target_sensitivity:.0%} not reachable, using {float(thr[0]):.3f}")
    return float(thr[0])

def evaluate(name, y, p, target_sensitivity=1 - MAX_UNDERTRIAGE, store=True):
    y = np.asarray(y)
    p = np.asarray(p)
    thr = pick_threshold(y, p, target_sensitivity)
    yhat = (p >= thr).astype(int)
    row = dict(sensitivity=recall_score(y, yhat),
               precision=precision_score(y, yhat, zero_division=0),
               f1=f1_score(y, yhat, zero_division=0),
               pr_auc=average_precision_score(y, p),
               roc_auc=roc_auc_score(y, p),
               brier=brier_score_loss(y, p),
               threshold=thr, alert_rate=float(yhat.mean()))
    if store:
        RESULTS[name] = row
    print(f"{name:32} sens={row['sensitivity']:.3f} prec={row['precision']:.3f} "
          f"PR-AUC={row['pr_auc']:.3f} ROC={row['roc_auc']:.3f} Brier={row['brier']:.3f} "
          f"thr={thr:.2f} alert={row['alert_rate']:.1%}")
    return row

def undertriage_alert_curve(y, p, n=40):
    y = np.asarray(y)
    p = np.asarray(p)
    rows = []
    for t in np.unique(np.quantile(p, np.linspace(0.5, 0.999, n))):
        yhat = (p >= t).astype(int)
        und = 1 - recall_score(y, yhat) if yhat.sum() else 1.0
        rows.append((float(t), float(und), float(yhat.mean())))
    return pd.DataFrame(rows, columns=["threshold", "undertriage", "alert_rate"])

def plot_diagnostics(name, y, p, thr, fname):
    y = np.asarray(y)
    p = np.asarray(p)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    prec, rec, _ = precision_recall_curve(y, p)
    ax[0].plot(rec, prec)
    ax[0].set_xlabel("recall")
    ax[0].set_ylabel("precision")
    ax[0].set_title(f"PR curve - {name}")
    try:
        fop, mpv = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax[1].plot(mpv, fop, "o-")
    except (ValueError, IndexError):
        ax[1].text(0.5, 0.5, "calibration N/A", ha="center", va="center")
    ax[1].plot([0, 1], [0, 1], "--", color="gray")
    ax[1].set_xlabel("predicted prob")
    ax[1].set_ylabel("observed freq")
    ax[1].set_title("Calibration")
    cm = confusion_matrix(y, (p >= thr).astype(int))
    ax[2].imshow(cm, cmap="Blues")
    for (r, c), v in np.ndenumerate(cm):
        ax[2].text(c, r, int(v), ha="center", va="center")
    ax[2].set_xticks([0, 1])
    ax[2].set_xticklabels(["low", "high"])
    ax[2].set_yticks([0, 1])
    ax[2].set_yticklabels(["low", "high"])
    ax[2].set_xlabel("predicted")
    ax[2].set_ylabel("true")
    ax[2].set_title(f"Confusion at thr={thr:.2f}")
    plt.tight_layout()
    plt.savefig(os.path.join(ASSETS, fname), dpi=90)
    plt.show()

def results_table():
    cols = ["sensitivity", "precision", "f1", "pr_auc", "roc_auc", "brier", "threshold", "alert_rate"]
    return pd.DataFrame(RESULTS).T[cols].round(3)

## 6. Structured model (vitals and intake)

How far do vitals and intake fields get on their own? We fit a gradient-boosted tree
(`HistGradientBoostingClassifier`, which handles missing values without imputation) and run it
twice: with and without the engineered scores flagged as leaky in section 2. The difference
shows how much of the performance comes from the generator's own severity score.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.ensemble import HistGradientBoostingClassifier

COST_RATIO = 8
CLASS_WEIGHT = {0: 1, 1: COST_RATIO}

exclude = set(["patient_id", TARGET, "high_risk", TEXT_COL, "cc_clean"] + LEAK_OUTCOME)
feat_cols = [c for c in train.columns if c not in exclude]
num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in feat_cols if c not in num_cols]

def fit_structured(label, drop_leaky):
    nc = [c for c in num_cols if not (drop_leaky and c in ENGINEERED_LEAKY)]
    pre = ColumnTransformer([
        ("num", "passthrough", nc),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         cat_cols)])
    clf = Pipeline([("pre", pre),
                    ("hgb", HistGradientBoostingClassifier(random_state=SEED, class_weight=CLASS_WEIGHT))])
    clf.fit(tr[nc + cat_cols], tr["high_risk"])
    p = clf.predict_proba(va[nc + cat_cols])[:, 1]
    evaluate(label, va["high_risk"], p)
    return clf, p

fit_structured("Structured (all features)", drop_leaky=False)
struct_clf, struct_p = fit_structured("Structured (no leaky score)", drop_leaky=True)

## 7. Text model: TF-IDF and logistic regression

This model sees only the chief complaint. Word 1-2 grams and character 3-5 grams go into a
logistic regression with balanced class weights. The character n-grams cope with
abbreviations and typos, and the coefficients show which terms the model relies on.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

union = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
])
Xtr = union.fit_transform(tr["cc_clean"])
Xva = union.transform(va["cc_clean"])
tfidf_lr = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, random_state=SEED)
tfidf_lr.fit(Xtr, tr["high_risk"])
tfidf_p = tfidf_lr.predict_proba(Xva)[:, 1]
evaluate("TF-IDF + LogReg (text)", va["high_risk"], tfidf_p)
plot_diagnostics("TF-IDF + LogReg", va["high_risk"], tfidf_p,
                 RESULTS["TF-IDF + LogReg (text)"]["threshold"], "diag_tfidf.png")

names = np.array(union.get_feature_names_out())
order = np.argsort(tfidf_lr.coef_[0])
strip_prefix = lambda i: names[i].replace("word__", "").replace("char__", "")
print("high-risk terms:", [strip_prefix(i) for i in order[::-1][:12]])
print("low-risk terms: ", [strip_prefix(i) for i in order[:12]])

### Stacked ensemble

A logistic regression combines the text and structured probabilities. Its training inputs are
out-of-fold predictions, so it never learns from scores the base models produced on rows they
were trained on.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict, StratifiedKFold

_oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
_struct_cols = [c for c in num_cols if c not in ENGINEERED_LEAKY] + cat_cols
tr_struct_p = cross_val_predict(clone(struct_clf), tr[_struct_cols], tr["high_risk"],
                                cv=_oof_cv, method="predict_proba")[:, 1]
_tfidf_pipe = Pipeline([("union", clone(union)), ("lr", clone(tfidf_lr))])
tr_tfidf_p = cross_val_predict(_tfidf_pipe, tr["cc_clean"], tr["high_risk"],
                               cv=_oof_cv, method="predict_proba")[:, 1]
tr_stack = np.column_stack([tr_tfidf_p, tr_struct_p])
va_stack = np.column_stack([tfidf_p, struct_p])

meta = LogisticRegression(random_state=SEED, class_weight={0: 1, 1: 10})
meta.fit(tr_stack, tr["high_risk"])
ensemble_p = meta.predict_proba(va_stack)[:, 1]
_ = evaluate("Ensemble (text + structured)", va["high_risk"], ensemble_p)

## 8. Transformer (Bio_ClinicalBERT)

Fine-tunes Bio_ClinicalBERT on the chief complaint with the Hugging Face `Trainer`. On Kaggle,
either attach the model as a dataset input (we used `aditidutta/bio-clinicalbert`) or turn
internet on, and use a GPU. Without a GPU or the `transformers` library the cell skips itself
and the rest of the notebook still runs. Set the environment variable `RUN_TRANSFORMER=0` to
skip it on purpose.

In [ ]:
RUN_TRANSFORMER = os.environ.get("RUN_TRANSFORMER", "1") == "1"
bert_p = None
try:
    import transformers
    _has_tf = True
except Exception:
    _has_tf = False

if not (RUN_TRANSFORMER and _has_tf and HAS_TORCH):
    print("transformer skipped: needs RUN_TRANSFORMER=1, transformers and torch")
else:
    try:
        from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                                  TrainingArguments, Trainer, DataCollatorWithPadding)
        from datasets import Dataset

        _local = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
                  if "competition" not in p.lower()]
        _local = [d for d in _local if any(k in d.lower() for k in ("bert", "clinical"))] or _local
        if _local:
            os.environ["HF_HUB_OFFLINE"] = "1"
            os.environ["TRANSFORMERS_OFFLINE"] = "1"
            MODEL_NAME = _local[0]
        else:
            MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
        MODEL_LABEL = "Bio_ClinicalBERT"
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        enc = lambda b: tok(b["text"], truncation=True, max_length=64)

        ds_tr = Dataset.from_dict({"text": tr["cc_clean"].tolist(),
                                   "label": tr["high_risk"].tolist()}).map(enc, batched=True, remove_columns=["text"])
        ds_va = Dataset.from_dict({"text": va["cc_clean"].tolist(),
                                   "label": va["high_risk"].tolist()}).map(enc, batched=True, remove_columns=["text"])
        collator = DataCollatorWithPadding(tok)

        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        args = TrainingArguments(output_dir="bert_out", num_train_epochs=3,
                                 per_device_train_batch_size=32, per_device_eval_batch_size=64,
                                 eval_strategy="epoch", seed=SEED, report_to=[], logging_steps=50,
                                 fp16=torch.cuda.is_available())

        common = dict(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va, data_collator=collator)
        try:
            trainer = Trainer(**common, processing_class=tok)
        except TypeError:
            trainer = Trainer(**common, tokenizer=tok)

        trainer.train()
        logits = trainer.predict(ds_va).predictions
        bert_p = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
        evaluate(f"{MODEL_LABEL} (text)", va["high_risk"], bert_p)
        plot_diagnostics(MODEL_LABEL, va["high_risk"], bert_p,
                         RESULTS[f"{MODEL_LABEL} (text)"]["threshold"], "diag_bert.png")
    except Exception as _e:
        bert_p = None
        print("transformer skipped:", _e)

In [ ]:
candidates = {"Ensemble (text + structured)": ensemble_p}
if bert_p is not None:
    candidates[f"{MODEL_LABEL} (text)"] = bert_p

primary_name = max(candidates, key=lambda name: RESULTS[name]["sensitivity"])
primary_p = candidates[primary_name]
primary_thr = pick_threshold(va["high_risk"], primary_p)
print(f"selected: {primary_name}, thr={primary_thr:.2f}")

## 9. Operating point and a vitals safety net

Every model here reaches a ROC-AUC near 1.0 on this synthetic data, so a stronger model is not
what matters. The decision rule is. We change it in two ways:

1. The threshold comes from an undertriage budget. We take the highest threshold that keeps
   undertriage at or below 2%, read off the tradeoff curve below. Because the classes separate
   cleanly, the alert rate stays close to the true share of high-risk patients, so fewer misses
   do not mean many more alerts.
2. Dangerous vital signs override the model. Any patient with GCS <= 8, SpO2 < 90, systolic
   BP < 90, heart rate > 130, respiratory rate > 30 or altered mental status is flagged
   high-risk whatever the model says. These cut-offs follow the danger thresholds behind NEWS2.
   The rule involves no machine learning, so staff know exactly when it fires.

In [ ]:
CRITICAL = {"gcs_total": ("<=", 8), "spo2": ("<", 90), "systolic_bp": ("<", 90),
            "heart_rate": (">", 130), "respiratory_rate": (">", 30)}

def vitals_override(df):
    flag = pd.Series(False, index=df.index)
    fired = {}
    for col, (op, thr) in CRITICAL.items():
        if col not in df.columns:
            continue
        s = df[col]
        m = {"<": s.lt(thr), "<=": s.le(thr), ">": s.gt(thr), ">=": s.ge(thr)}[op].fillna(False)
        fired[col] = int(m.sum())
        flag = flag | m
    if "mental_status_triage" in df.columns:
        m = df["mental_status_triage"].astype(str).str.lower().eq("altered")
        fired["mental_status_altered"] = int(m.sum())
        flag = flag | m
    return flag.to_numpy(), fired

def undertriage_rate(y, pred):
    y = np.asarray(y)
    pred = np.asarray(pred)
    pos = y == 1
    return float((pred[pos] == 0).mean()) if pos.any() else 0.0

y_va = va["high_risk"].to_numpy()
model_pred = (primary_p >= primary_thr).astype(int)
ov_flag, fired = vitals_override(va)
final_pred = ((model_pred == 1) | ov_flag).astype(int)
rescued = int(((y_va == 1) & (model_pred == 0) & ov_flag).sum())

print("override fired:", fired)
print(pd.DataFrame({"undertriage": [undertriage_rate(y_va, model_pred), undertriage_rate(y_va, final_pred)],
                    "alert_rate": [model_pred.mean(), final_pred.mean()]},
                   index=["model only", "model + vitals override"]).round(4).to_string())
print(f"override caught {rescued} missed high-risk patients for "
      f"{int(((model_pred == 0) & ov_flag).sum())} extra alerts")

curve = undertriage_alert_curve(y_va, primary_p)
plt.figure(figsize=(6, 4))
plt.plot(curve["alert_rate"], curve["undertriage"], "-o", ms=3)
plt.axhline(MAX_UNDERTRIAGE, ls="--", color="#E45756", label=f"{MAX_UNDERTRIAGE:.0%} budget")
plt.xlabel("alert rate (staff workload)")
plt.ylabel("undertriage rate (missed high-risk)")
plt.title("Undertriage vs alert-rate tradeoff")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "undertriage_tradeoff.png"), dpi=90)
plt.show()

## 10. Predict, but escalate when unsure

A triage tool has to behave safely on input that looks nothing like its training data. The
model below always predicts an ESI level. A gate then decides whether that prediction can
stand or should go to a clinician:

- Confidence: the top class probability must clear a floor set so that about 5% of
  validation cases are escalated.
- Unfamiliar text: if more than half the words never appeared in training (a new or garbled
  complaint), the case is escalated whatever the prediction.
- Fixed overrides: a list of red-flag terms (for example "cardiac arrest" or "not breathing")
  and the vital sign thresholds from section 9 force a high-risk decision.

When the gate is not satisfied the tool still shows its best guess, for example "predicts
ESI 2, but this complaint is unlike the training data, so escalate to a clinician".

In [ ]:
esi_model = Pipeline([
    ("tfidf", FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5))])),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
]).fit(tr["cc_clean"], tr[TARGET])
ESI_CLASSES = [int(c) for c in esi_model.classes_]
HI_IDX = [i for i, c in enumerate(ESI_CLASSES) if c <= 2]

FULL_VOCAB = set(CountVectorizer(token_pattern=r"[a-z]+").fit(tr["cc_clean"]).vocabulary_.keys())
RED_FLAGS = ["decapitat", "cardiac arrest", "not breathing", "unresponsive", "anaphylaxis",
             "exsanguinat", "active haemorrhage", "active hemorrhage", "gunshot", "amputation",
             "agonal", "asystole", "stab wound", "hanging", "drowning"]

def _oov_ratio(clean):
    words = [w for w in clean.split() if w.isalpha()]
    return 1.0 if not words else sum(w not in FULL_VOCAB for w in words) / len(words)

def _vitals_override_dict(vitals):
    reasons = []
    if vitals:
        for col, (op, thr) in CRITICAL.items():
            try:
                v = float(vitals.get(col))
            except (TypeError, ValueError):
                continue
            if {"<": v < thr, "<=": v <= thr, ">": v > thr, ">=": v >= thr}[op]:
                reasons.append(f"{col} {op} {thr}")
        if str(vitals.get("mental_status_triage", "")).lower() == "altered":
            reasons.append("altered mental status")
    return reasons

_va_proba = esi_model.predict_proba(va["cc_clean"])
CONF_THR = float(np.quantile(_va_proba.max(1), 0.05))
OOV_THR = 0.50
ESI_THR = pick_threshold(va["high_risk"].to_numpy(), _va_proba[:, HI_IDX].sum(1))
print(f"gate: confidence >= {CONF_THR:.2f}, unseen words <= {OOV_THR:.0%}, high-risk thr = {ESI_THR:.2f}")

def triage_assess(text, vitals=None):
    clean = clean_text(text)
    proba = esi_model.predict_proba([clean])[0]
    pdist = {c: float(p) for c, p in zip(ESI_CLASSES, proba)}
    pred_esi = max(pdist, key=pdist.get)
    conf = pdist[pred_esi]
    p_high = sum(pdist[c] for c in ESI_CLASSES if c <= 2)
    oov = _oov_ratio(clean)
    overrides = [f"red-flag: {k}" for k in RED_FLAGS if k in clean] + _vitals_override_dict(vitals)

    if overrides:
        decision, finalized, reason = "HIGH-RISK", True, "override (" + ", ".join(overrides) + ")"
    elif not (oov <= OOV_THR and conf >= CONF_THR):
        decision, finalized = "ESCALATE", False
        reason = "outside training distribution" if oov > OOV_THR else "low model confidence"
    else:
        decision = "HIGH-RISK" if p_high >= ESI_THR else "LOWER ACUITY"
        finalized, reason = True, "confident, in distribution"
    return dict(pred_esi=pred_esi, conf=conf, p_high=p_high, oov=oov,
                decision=decision, finalized=finalized, reason=reason)

for t in ["chest pain radiating to left arm", "mild sore throat", "decapitated",
          "patient reports glorptang fzzle since noon"]:
    r = triage_assess(t)
    print(f"{t[:44]:46} ESI {r['pred_esi']} {r['decision']:13} {r['reason']}")

_defer = va["cc_clean"].map(lambda c: not triage_assess(c)["finalized"]).mean()
print(f"escalation rate on validation: {_defer:.1%}")

_samp = va.sample(min(800, len(va)), random_state=SEED)
_c = esi_model.predict_proba(_samp["cc_clean"]).max(1)
_o = _samp["cc_clean"].map(_oov_ratio).to_numpy()
_fin = (_c >= CONF_THR) & (_o <= OOV_THR)
plt.figure(figsize=(6, 4))
plt.scatter(_c[_fin], _o[_fin], s=8, color="#4C78A8", alpha=.6, label="finalize")
plt.scatter(_c[~_fin], _o[~_fin], s=14, color="#E45756", alpha=.7, label="escalate")
plt.axvline(CONF_THR, ls="--", color="gray")
plt.axhline(OOV_THR, ls="--", color="gray")
plt.xlabel("model confidence (max class prob)")
plt.ylabel("share of unseen words")
plt.title("Escalation gate (validation)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "ood_gate.png"), dpi=90)
plt.show()

## 11. Ordinal and safety metrics

ESI is an ordered scale, so we also report the metrics used for it in the triage literature:
quadratic-weighted kappa (QWK) and the share of predictions within one level. The safety rates
are named explicitly. Dangerous undertriage is a true ESI 1-2 predicted as 3 or higher. A
critical miss is a true ESI 1 predicted as 3 or higher. Overtriage is a true ESI 3-5 predicted
as 1 or 2. A baseline that bins the NEWS2 score into five levels is shown for comparison.

In [ ]:
from sklearn.metrics import cohen_kappa_score

_p = esi_model.predict_proba(va["cc_clean"])
_pred = np.array(ESI_CLASSES)[_p.argmax(1)]
_true = va[TARGET].to_numpy()

def acuity_report(name, true, pred):
    true, pred = np.asarray(true), np.asarray(pred)
    dang = ((true <= 2) & (pred >= 3)).sum() / max((true <= 2).sum(), 1)
    crit = ((true == 1) & (pred >= 3)).sum() / max((true == 1).sum(), 1)
    over = ((true >= 3) & (pred <= 2)).sum() / max((true >= 3).sum(), 1)
    try:
        _qwk = cohen_kappa_score(true, pred, weights="quadratic")
        _lk = cohen_kappa_score(true, pred, weights="linear")
    except Exception:
        _qwk = _lk = float("nan")
    row = dict(QWK=_qwk, linear_kappa=_lk,
               within_1=np.mean(np.abs(true - pred) <= 1),
               mae_acuity=np.mean(np.abs(true - pred)),
               dangerous_undertriage=dang, critical_miss=crit, overtriage=over)
    print(f"{name:22} QWK={row['QWK']:.3f} within1={row['within_1']:.3f} MAE={row['mae_acuity']:.3f} "
          f"dangerous undertriage={dang:.2%} critical miss={crit:.2%} overtriage={over:.2%}")
    return row

acuity_report("ESI model (text)", _true, _pred)

if "news2_score" in va.columns:
    try:
        _b = pd.qcut(va["news2_score"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
        acuity_report("NEWS2 cutoff baseline", _true, _b.to_numpy())
    except Exception as _e:
        print("NEWS2 baseline skipped:", _e)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, c in zip(ax, [1, 2]):
    if c in ESI_CLASSES:
        pc = _p[:, ESI_CLASSES.index(c)]
        try:
            fop, mpv = calibration_curve((_true == c).astype(int), pc, n_bins=8, strategy="quantile")
            a.plot(mpv, fop, "o-")
        except Exception:
            a.text(0.5, 0.5, "N/A", ha="center")
        a.plot([0, 1], [0, 1], "--", color="gray")
        a.set_title(f"Reliability: ESI {c}")
        a.set_xlabel("predicted")
        a.set_ylabel("observed")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "reliability_per_class.png"), dpi=90)
plt.show()

## 12. Conformal prediction sets

Split conformal prediction turns the ESI model's probabilities into a set of levels that
contains the true level about 90% of the time (alpha = 0.10), with no assumption about the
data distribution. The set grows on hard cases. A patient whose set spans more than one level
is a natural candidate for a senior review.

In [ ]:
_cls = ESI_CLASSES
_pos = {c: i for i, c in enumerate(_cls)}
_vac = va[va[TARGET].isin(_cls)]
_P = esi_model.predict_proba(_vac["cc_clean"])
_y = _vac[TARGET].to_numpy()
_truecol = np.array([_pos[c] for c in _y])
_idx = np.arange(len(_vac))
try:
    _cal, _tst = train_test_split(_idx, test_size=0.5, random_state=SEED, stratify=_y)
except ValueError:
    _cal, _tst = train_test_split(_idx, test_size=0.5, random_state=SEED)
_scores = 1 - _P[_idx, _truecol]
alpha = 0.10
_n = len(_cal)
# finite-sample corrected quantile level, capped at 1
_level = min(np.ceil((_n + 1) * (1 - alpha)) / _n, 1.0)
_qhat = np.quantile(_scores[_cal], _level, method="higher")
_sets = _P[_tst] >= (1 - _qhat)
_cover = _sets[np.arange(len(_tst)), _truecol[_tst]].mean()
_size = _sets.sum(1)
print(f"coverage={_cover:.3f} (target {1 - alpha:.2f}), mean set size={_size.mean():.2f}, "
      f"multi-level sets={(_size > 1).mean():.1%}, empty sets={(_size == 0).mean():.1%}")

plt.figure(figsize=(5, 3))
plt.hist(_size, bins=np.arange(0, len(_cls) + 2) - 0.5, color="#4C78A8", edgecolor="white")
plt.xlabel("prediction set size")
plt.ylabel("patients")
plt.title("Conformal set sizes")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "conformal_sets.png"), dpi=90)
plt.show()

## 13. Validation checks

1. GroupKFold by chief complaint. No complaint wording is shared between folds. If accuracy
   drops sharply compared with ordinary stratified folds, the text model is memorising
   phrases rather than generalising.
2. Ambiguous complaints. How often does the same complaint appear with more than one ESI level?
3. Outcomes by predicted level. Admission rate and length of stay should fall as predicted
   acuity becomes less urgent. Outcomes are used here only to check the model, never as features.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score

_txt = Pipeline([("tfidf", FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5))])),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])
if train[TEXT_COL].nunique() >= 2:
    _grp = train[TEXT_COL].fillna("__na__")
    _ns = int(min(5, _grp.nunique()))
    _skf = cross_val_score(_txt, train["cc_clean"], train[TARGET],
                           cv=StratifiedKFold(_ns, shuffle=True, random_state=SEED), scoring="accuracy").mean()
    _gkf = cross_val_score(_txt, train["cc_clean"], train[TARGET], cv=GroupKFold(_ns),
                           groups=_grp, scoring="accuracy").mean()
    print(f"text accuracy: StratifiedKFold={_skf:.3f}, GroupKFold by complaint={_gkf:.3f} "
          f"(change {_gkf - _skf:+.3f})")
    _amb = train.groupby(TEXT_COL)[TARGET].nunique()
    print(f"{(_amb == 1).mean():.1%} of distinct complaints map to one ESI level, "
          f"{int((_amb > 1).sum())} map to several")

_o = va.copy()
_o["pred_esi"] = _pred
if "disposition" in _o.columns:
    _o["admit"] = _o["disposition"].astype(str).str.lower().isin(["admit", "admitted", "icu"]).astype(int)
    _cols = {"n": ("pred_esi", "size"), "admit_rate": ("admit", "mean")}
    if "ed_los_hours" in _o.columns:
        _cols["mean_los_h"] = ("ed_los_hours", "mean")
    _tab = _o.groupby("pred_esi").agg(**_cols).round(3)
    print(_tab.to_string())
    print("admit rate falls with predicted ESI:", _tab["admit_rate"].is_monotonic_decreasing)

## 14. Model comparison

All numbers are on the held-out validation split. Inference time matters for deployment: the
TF-IDF model runs in well under a millisecond per patient on a CPU, while a transformer needs
far more compute at the triage desk.

In [ ]:
print(results_table().to_string())

def latency_ms(fn, X, n=3):
    if len(X) == 0:
        return float("nan")
    t = time.perf_counter()
    for _ in range(n):
        fn(X)
    return (time.perf_counter() - t) / n / len(X) * 1000

lat_tfidf = latency_ms(lambda x: tfidf_lr.predict_proba(union.transform(x)), va["cc_clean"])
lat_struct = latency_ms(lambda x: struct_clf.predict_proba(x), va[_struct_cols])
print(f"latency per record: TF-IDF {lat_tfidf:.3f} ms, structured {lat_struct:.3f} ms")

ax = results_table()[["sensitivity", "precision", "pr_auc"]].plot(kind="bar", figsize=(10, 4))
ax.set_title("Model comparison (validation)")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "model_comparison.png"), dpi=90)
plt.show()

## 15. Error analysis: who do we miss?

The dangerous errors are high-risk patients that the final decision (model plus vitals
override) still leaves unflagged. We pull them out to see why they were missed.

In [ ]:
va_eval = va.copy()
va_eval["prob"] = primary_p
va_eval["pred"] = final_pred
missed = va_eval[(va_eval["high_risk"] == 1) & (va_eval["pred"] == 0)]
n_high = int(va_eval["high_risk"].sum())
blank = missed[TEXT_COL].fillna("").str.strip().eq("")
print(f"missed {len(missed)} of {n_high} high-risk patients ({len(missed) / max(n_high, 1):.1%}), "
      f"{int(blank.sum())} with a blank complaint")
print(missed[[TARGET, "prob", TEXT_COL]].head(8).round(2).to_string(index=False))

## 16. Fairness: undertriage by patient group

A good overall sensitivity can hide a group that is missed more often. For each demographic
axis we compute the undertriage rate (the share of truly high-risk patients left unflagged)
with a 95% bootstrap interval. A group whose whole interval sits above the overall rate is
marked `exceeds_overall`. Subgroups are small on a 20% validation split, so the intervals are
wide.

In [ ]:
def undertriage_by(axis, df=va_eval):
    d = df[df["high_risk"] == 1]
    g = d.groupby(axis).apply(
        lambda x: pd.Series({"n_high_risk": len(x),
                             "missed": int((x["pred"] == 0).sum()),
                             "undertriage_rate": (x["pred"] == 0).mean()}),
        include_groups=False)
    return g.sort_values("undertriage_rate", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, axis in zip(ax, [c for c in ["language", "insurance_type"] if c in FAIRNESS_AXES][:2]):
    t = undertriage_by(axis)
    a.bar(t.index.astype(str), t["undertriage_rate"], color="#E45756")
    a.set_title(f"Undertriage rate by {axis}")
    a.set_ylabel("FNR among high-risk")
    a.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "fairness_undertriage.png"), dpi=90)
plt.show()

In [ ]:
def bootstrap_undertriage_ci(df, axis, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    d = df[df["high_risk"] == 1]
    overall = float((d["pred"] == 0).mean()) if len(d) else float("nan")
    rows = []
    for grp, g in d.groupby(axis):
        miss = (g["pred"] == 0).to_numpy().astype(float)
        n = len(miss)
        if n == 0:
            rows.append({axis: grp, "n": 0, "undertriage": np.nan,
                         "ci_lo": np.nan, "ci_hi": np.nan, "exceeds_overall": False})
            continue
        boot = miss[rng.integers(0, n, size=(n_boot, n))].mean(axis=1)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append({axis: grp, "n": n, "undertriage": float(miss.mean()),
                     "ci_lo": float(lo), "ci_hi": float(hi),
                     "exceeds_overall": bool(lo > overall)})
    out = pd.DataFrame(rows).sort_values("undertriage", ascending=False)
    return out, overall

ci_tables = {}
for axis in FAIRNESS_AXES:
    ci_tables[axis], overall = bootstrap_undertriage_ci(va_eval, axis)
    print(f"{axis} (overall {overall:.3f})")
    print(ci_tables[axis].round(3).to_string(index=False))

fig, axx = plt.subplots(1, len(FAIRNESS_AXES), figsize=(4 * len(FAIRNESS_AXES), 3.4))
for _ax, _axis in zip(np.atleast_1d(axx), FAIRNESS_AXES):
    _tbl = ci_tables[_axis]
    _x = _tbl[_axis].astype(str).tolist()
    _y = _tbl["undertriage"].to_numpy()
    _err = np.vstack([_y - _tbl["ci_lo"].to_numpy(), _tbl["ci_hi"].to_numpy() - _y])
    _ax.errorbar(range(len(_x)), _y, yerr=_err, fmt="o", color="#E45756", capsize=4)
    _ax.axhline(overall, ls="--", color="gray")
    _ax.set_xticks(range(len(_x)))
    _ax.set_xticklabels(_x, rotation=30, ha="right")
    _ax.set_title(f"Undertriage by {_axis} (95% CI)")
    _ax.set_ylabel("FNR among high-risk")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "undertriage_ci.png"), dpi=90)
plt.show()

### Significance, intersections and complaint systems

A chi-square test per axis checks whether the differences between groups are larger than
chance. We also look at age group crossed with language, where disadvantages can compound,
and at undertriage by the body system of the complaint. If nurse IDs are present, an ANOVA
compares acuity levels assigned by different nurses.

In [ ]:
from scipy.stats import chi2_contingency, f_oneway

_high = va_eval[va_eval["high_risk"] == 1]
for axis in [a for a in FAIRNESS_AXES if a in _high.columns]:
    ct = pd.crosstab(_high[axis], _high["pred"] == 0)
    if ct.shape[0] > 1 and ct.shape[1] > 1:
        try:
            chi2, p, _, _ = chi2_contingency(ct)
            print(f"{axis:14} chi2={chi2:.1f} p={p:.4f}{' significant' if p < 0.05 else ''}")
        except Exception as _e:
            print(f"{axis}: chi-square skipped ({_e})")

if {"age_group", "language"}.issubset(va_eval.columns):
    inter = (_high.assign(missed=(_high["pred"] == 0).astype(int))
                  .groupby(["age_group", "language"])["missed"].mean().unstack())
    print((inter * 100).round(1).to_string())

if "chief_complaint_system" in _high.columns:
    sysu = (_high.assign(missed=(_high["pred"] == 0).astype(int))
                 .groupby("chief_complaint_system")["missed"].agg(["mean", "size"])
                 .sort_values("mean", ascending=False))
    print(sysu.round(3).to_string())

if "triage_nurse_id" in train.columns:
    grp = [g[TARGET].to_numpy() for _, g in train.groupby("triage_nurse_id") if len(g) >= 30]
    if len(grp) >= 2:
        F, p = f_oneway(*grp)
        print(f"inter-nurse acuity ANOVA: F={F:.1f}, p={p:.3g}")

## 17. Feature importance and keyword weights

Permutation importance is measured on held-out rows, so it reflects predictive value on new
patients rather than how often a tree happened to split on a feature.

In [ ]:
from sklearn.inspection import permutation_importance

def _feat_family(c):
    if c.startswith("ix_"):
        return "interaction"
    if c.startswith("hx_"):
        return "comorbidity"
    if c in ("qsofa", "sirs", "charlson_index", "rate_pressure_product", "mod_shock_index",
             "spo2_rr_ratio", "peds_shock_index", "news2_score", "shock_index"):
        return "clinical score"
    if c in ("systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate", "temperature_c",
             "spo2", "gcs_total", "pain_score", "mean_arterial_pressure", "n_vitals_missing", "age"):
        return "vital"
    if c.endswith(("_sin", "_cos")):
        return "time"
    return "other"

_FAM = {"vital": "#4C78A8", "clinical score": "#F58518", "interaction": "#54A24B",
        "comorbidity": "#B279A2", "time": "#9D755D", "other": "#BAB0AC"}

_samp = va.sample(min(4000, len(va)), random_state=SEED)
_pi = permutation_importance(struct_clf, _samp[_struct_cols], _samp["high_risk"],
                             scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
_imp = (pd.DataFrame({"feature": _struct_cols, "imp": _pi.importances_mean, "sd": _pi.importances_std})
        .sort_values("imp", ascending=False))
_imp["family"] = _imp["feature"].map(_feat_family)
_top = _imp.head(15).iloc[::-1]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].barh(_top["feature"], _top["imp"], xerr=_top["sd"], color=[_FAM[f] for f in _top["family"]])
ax[0].set_title("Structured model: permutation importance (val ROC-AUC drop)")
ax[0].set_xlabel("mean importance +/- sd")
ax[0].legend([plt.Rectangle((0, 0), 1, 1, color=_FAM[f]) for f in _FAM], list(_FAM),
             fontsize=8, loc="lower right")

_coef = tfidf_lr.coef_[0]
_w = [(i, names[i]) for i in np.argsort(_coef)[::-1] if names[i].startswith("word__")][:12][::-1]
ax[1].barh([n.replace("word__", "") for _, n in _w], [_coef[i] for i, _ in _w], color="#E45756")
ax[1].set_title("Text model: top high-risk keyword weights")
ax[1].set_xlabel("coefficient toward high-risk")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "feature_importance.png"), dpi=90)
plt.show()

_fam_share = _imp.groupby("family")["imp"].sum().clip(lower=0)
print((_fam_share / _fam_share.sum() * 100).sort_values(ascending=False).round(1).to_string())

## 18. Cost-sensitive decision rule

Missing a truly high-acuity patient is far worse than overtriage. Instead of taking the most
likely class, this rule picks the ESI level with the lowest expected cost under a cost matrix
that can be read and changed. We also check that the result holds when the undertriage
penalty is varied.

In [ ]:
_classes = np.array(ESI_CLASSES)
_P = esi_model.predict_proba(va["cc_clean"])
_ytrue = va[TARGET].to_numpy().astype(int)

def _cost_matrix(ut_crit=10, ut_mid=2, ot=1):
    K = len(_classes)
    C = np.zeros((K, K))
    for ti, t in enumerate(_classes):
        for pj, p in enumerate(_classes):
            gap = abs(int(p) - int(t))
            if p > t:
                C[ti, pj] = gap * (ut_crit if t <= 2 else ut_mid)
            elif p < t:
                C[ti, pj] = gap * ot
    return C

def _crit(pred):
    m = _ytrue <= 2
    return float((pred[m] >= 4).mean()) if m.any() else 0.0

def _anyut(pred):
    m = _ytrue <= 2
    return float((pred[m] > 2).mean()) if m.any() else 0.0

_naive = _classes[_P.argmax(1)]
_cost = _classes[(_P @ _cost_matrix()).argmin(1)]
print(pd.DataFrame({"accuracy": [(pr == _ytrue).mean() for pr in (_naive, _cost)],
                    "ESI 1-2 to 4-5": [_crit(pr) for pr in (_naive, _cost)],
                    "ESI 1-2 to 3-5": [_anyut(pr) for pr in (_naive, _cost)]},
                   index=["argmax", "cost-sensitive"]).round(4).to_string())

_mults = [5, 10, 15]
_critv = [_crit(_classes[(_P @ _cost_matrix(ut_crit=m)).argmin(1)]) for m in _mults]
_accv = [(_classes[(_P @ _cost_matrix(ut_crit=m)).argmin(1)] == _ytrue).mean() for m in _mults]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(["argmax", "cost-sensitive"], [_crit(_naive), _crit(_cost)], color=["#BAB0AC", "#E45756"])
ax[0].set_title("Critical undertriage (true ESI 1-2 -> ESI 4-5)")
ax[0].set_ylabel("rate")
ax[1].plot(_mults, _critv, "-o", label="critical undertriage")
ax[1].plot(_mults, _accv, "-s", label="accuracy")
ax[1].set_xlabel("undertriage cost multiplier")
ax[1].set_title("Sensitivity to the cost setting")
ax[1].legend()
ax[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "cost_sensitive.png"), dpi=90)
plt.show()

## 19. Missing vitals are not missing at random

Missing vital signs cluster by acuity level and track undertriage risk, so a missing
measurement carries information. That is why the structured model keeps `n_vitals_missing`
instead of imputing the gaps away.

In [ ]:
_vits = [c for c in ["systolic_bp", "diastolic_bp", "respiratory_rate", "temperature_c", "heart_rate",
                     "spo2"] if c in train.columns]
_md = pd.DataFrame({c: train[c].isna().groupby(train[TARGET]).mean() for c in _vits})
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
_md.plot(kind="bar", ax=ax[0])
ax[0].set_title("Vital missingness by ESI level")
ax[0].set_xlabel("ESI level")
ax[0].set_ylabel("fraction missing")
ax[0].legend(fontsize=7, ncol=2)
ax[0].tick_params(axis="x", rotation=0)
_vp = np.array(ESI_CLASSES)[esi_model.predict_proba(va["cc_clean"]).argmax(1)]
_ut = (_vp > va[TARGET].to_numpy()).astype(int)
_nm = va["n_vitals_missing"].clip(upper=3).to_numpy() if "n_vitals_missing" in va.columns else np.zeros(len(va), int)
_rate = pd.DataFrame({"nmiss": _nm, "ut": _ut}).groupby("nmiss")["ut"].mean()
ax[1].bar(_rate.index.astype(str), _rate.values, color="#F58518")
ax[1].set_title("Undertriage rate by number of missing vitals")
ax[1].set_xlabel("missing vitals (3 = 3 or more)")
ax[1].set_ylabel("undertriage rate")
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "mnar.png"), dpi=90)
plt.show()

_base = _rate.get(0, float("nan"))
print(pd.DataFrame({"undertriage": _rate,
                    "vs_none_missing": _rate / _base if _base > 0 else np.nan}).round(3).to_string())

## 20. Where the label comes from

A feature-source ablation compares three-fold accuracy for the triage-time features alone, the
complaint text alone, and the triage-time features plus the pre-computed scores. A
text-shuffle control pairs each complaint with a random patient. If accuracy falls to the
majority-class baseline, the text model was learning from the complaint itself.

In [ ]:
_ss = train.sample(min(6000, len(train)), random_state=SEED)
_cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
_honest = [c for c in num_cols if c not in ENGINEERED_LEAKY]
_leaky = [c for c in ("news2_score", "shock_index") if c in num_cols]

def _struct_pipe(cols):
    pre = ColumnTransformer([
        ("num", "passthrough", [c for c in cols if c in num_cols]),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         [c for c in cols if c in cat_cols])])
    return Pipeline([("pre", pre), ("hgb", HistGradientBoostingClassifier(random_state=SEED))])

_txtp = Pipeline([("tfidf", FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5))])),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])

_ablation = {"Triage-time features": (cross_val_score(_struct_pipe(_honest + cat_cols), _ss[_honest + cat_cols],
                                                      _ss[TARGET], cv=_cv, scoring="accuracy").mean(), "#54A24B"),
             "Text only": (cross_val_score(_txtp, _ss["cc_clean"], _ss[TARGET], cv=_cv,
                                           scoring="accuracy").mean(), "#4C78A8")}
if _leaky:
    _ablation["Features + leaky scores"] = (cross_val_score(_struct_pipe(num_cols + cat_cols), _ss[num_cols + cat_cols],
                                                            _ss[TARGET], cv=_cv, scoring="accuracy").mean(), "#E45756")

_shuf = _ss["cc_clean"].sample(frac=1.0, random_state=SEED).to_numpy()
_acc_shuf = cross_val_score(_txtp, pd.Series(_shuf, index=_ss.index), _ss[TARGET], cv=_cv, scoring="accuracy").mean()
_baseacc = _ss[TARGET].value_counts(normalize=True).max()

plt.figure(figsize=(8, 4))
plt.bar(list(_ablation), [v for v, _ in _ablation.values()], color=[c for _, c in _ablation.values()])
plt.axhline(_baseacc, ls="--", color="gray", label=f"majority-class baseline ({_baseacc:.2f})")
plt.ylabel("3-fold accuracy")
plt.title("Feature-source ablation")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, "leakage_ablation.png"), dpi=90)
plt.show()
print({k: round(float(v), 3) for k, (v, _) in _ablation.items()})
print(f"shuffled text accuracy {_acc_shuf:.3f}, majority baseline {_baseacc:.3f}")

## 21. External validation on real emergency department data (NHAMCS 2018-2022)

The competition data is synthetic, so we repeat the main text model on real visits from the
CDC's National Hospital Ambulatory Medical Care Survey, which records the triage level and the
chief complaint. The schemas differ, so the same architecture is re-fit on the real data
rather than transferring the trained weights. On Kaggle, attach the `reaper0ai/nhamcs-2018-22`
dataset; locally, set `NHAMCS_DIR` to a folder holding the CSV. Without it, sections 21 to 23
skip themselves.

In [ ]:
EXT = None

def _find_nhamcs():
    for r in ["/kaggle/input", os.environ.get("NHAMCS_DIR", "")]:
        if not r:
            continue
        for p in glob.glob(os.path.join(r, "**", "*.csv"), recursive=True):
            try:
                if "target_triage_acuity" in pd.read_csv(p, nrows=3).columns:
                    return p
            except Exception:
                pass
    return None

try:
    _path = _find_nhamcs()
    if _path is None:
        print("NHAMCS data not found, external validation skipped")
    else:
        real = pd.read_csv(_path).dropna(subset=["target_triage_acuity", "chief_complaint_text"])
        real["acuity"] = real["target_triage_acuity"].astype(int)
        real["cc_clean"] = real["chief_complaint_text"].astype(str).map(clean_text)
        real["hr"] = (real["acuity"] <= 2).astype(int)
        print(f"{len(real):,} real visits, acuity counts {real['acuity'].value_counts().sort_index().to_dict()}")

        rtr, rte = train_test_split(real, test_size=0.25, stratify=real["acuity"], random_state=SEED)
        txt = Pipeline([("tfidf", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5))])),
            ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
        txt.fit(rtr["cc_clean"], rtr["acuity"])
        _rcls = [int(c) for c in txt.classes_]
        proba = txt.predict_proba(rte["cc_clean"])
        pred = np.array(_rcls)[proba.argmax(1)]
        phigh = proba[:, [i for i, c in enumerate(_rcls) if c <= 2]].sum(1)

        real_qwk = cohen_kappa_score(rte["acuity"], pred, weights="quadratic")
        real_roc = roc_auc_score(rte["hr"], phigh) if rte["hr"].nunique() > 1 else float("nan")
        real_ut = float((pred[rte["hr"].to_numpy() == 1] > 2).mean())

        _sp = esi_model.predict_proba(va["cc_clean"])
        _spred = np.array(ESI_CLASSES)[_sp.argmax(1)]
        syn_qwk = cohen_kappa_score(va[TARGET], _spred, weights="quadratic")
        syn_roc = roc_auc_score((va[TARGET].to_numpy() <= 2).astype(int), _sp[:, HI_IDX].sum(1))
        print(f"real:      QWK={real_qwk:.3f} high-risk ROC-AUC={real_roc:.3f} undertriage ESI 1-2={real_ut:.1%}")
        print(f"synthetic: QWK={syn_qwk:.3f} high-risk ROC-AUC={syn_roc:.3f}")

        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].bar(["synthetic", "real NHAMCS"], [syn_qwk, real_qwk], color=["#BAB0AC", "#4C78A8"])
        ax[0].axhspan(0.6, 0.8, color="#54A24B", alpha=.15, label="human inter-rater range")
        ax[0].set_title("Quadratic-weighted kappa: synthetic vs real")
        ax[0].set_ylabel("QWK")
        ax[0].legend(fontsize=8)
        ax[1].bar(["synthetic", "real NHAMCS"], [syn_roc, real_roc], color=["#BAB0AC", "#4C78A8"])
        ax[1].set_title("High-risk ROC-AUC: synthetic vs real")
        ax[1].set_ylabel("ROC-AUC")
        plt.tight_layout()
        plt.savefig(os.path.join(ASSETS, "external_validation.png"), dpi=90)
        plt.show()

        EXT = {"rte": rte, "phigh": phigh, "pred": pred}
except Exception as _e:
    EXT = None
    print("external validation skipped:", _e)

## 22. Decision-curve analysis on the real data

Net benefit of the high-risk flag compared with flagging everyone or no one, across a range of
threshold probabilities (Vickers and Elkin, 2006). A flag is useful where its curve sits above
both reference lines.

In [ ]:
if EXT is None:
    print("decision curve skipped: no external data")
else:
    y = EXT["rte"]["hr"].to_numpy()
    p = EXT["phigh"]
    n = len(y)
    prev = y.mean()
    ts = np.linspace(0.02, 0.6, 50)
    nb_model, nb_all = [], []
    for t in ts:
        flag = p >= t
        tp = float(np.sum(flag & (y == 1)))
        fp = float(np.sum(flag & (y == 0)))
        w = t / (1 - t)
        nb_model.append(tp / n - fp / n * w)
        nb_all.append(prev - (1 - prev) * w)
    plt.figure(figsize=(7, 4.5))
    plt.plot(ts, nb_model, label="model (flag high-risk)", color="#4C78A8", lw=2)
    plt.plot(ts, nb_all, label="flag all", color="#F58518", ls="--")
    plt.axhline(0, label="flag none", color="gray", ls=":")
    plt.xlabel("threshold probability")
    plt.ylabel("net benefit")
    plt.title("Decision-curve analysis (real NHAMCS)")
    plt.legend()
    plt.grid(alpha=.3)
    plt.ylim(min(0, min(nb_model)) - 0.01, max(nb_model) + 0.02)
    plt.tight_layout()
    plt.savefig(os.path.join(ASSETS, "decision_curve.png"), dpi=90)
    plt.show()

## 23. Calibration and fairness on the real data

Reliability before and after isotonic recalibration, the Brier skill score against a
prevalence-only forecast, and undertriage by subgroup, all on the real NHAMCS visits.

In [ ]:
from sklearn.isotonic import IsotonicRegression

if EXT is None:
    print("calibration skipped: no external data")
else:
    y = EXT["rte"]["hr"].to_numpy()
    p = EXT["phigh"]
    ci, ei = train_test_split(np.arange(len(y)), test_size=0.5, random_state=SEED, stratify=y)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p[ci], y[ci])
    ye, praw, pcal = y[ei], p[ei], iso.predict(p[ei])
    bb = brier_score_loss(ye, np.full_like(praw, ye.mean()))
    skill = lambda b: (1 - b / bb) if bb > 0 else float("nan")
    b_raw, b_cal = brier_score_loss(ye, praw), brier_score_loss(ye, pcal)
    print(f"Brier raw={b_raw:.3f} (skill {skill(b_raw):+.2f}), isotonic={b_cal:.3f} (skill {skill(b_cal):+.2f})")

    def _rel(pr):
        b = np.clip((pr * 10).astype(int), 0, 9)
        xs, ys = [], []
        for k in range(10):
            m = b == k
            if m.sum() > 25:
                xs.append(pr[m].mean())
                ys.append(ye[m].mean())
        return xs, ys

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot([0, 1], [0, 1], "--", color="gray")
    xr, yr = _rel(praw)
    xc, yc = _rel(pcal)
    ax[0].plot(xr, yr, "-o", label="raw", color="#E45756")
    ax[0].plot(xc, yc, "-s", label="isotonic", color="#4C78A8")
    ax[0].set_xlabel("predicted P(high-risk)")
    ax[0].set_ylabel("observed")
    ax[0].set_title("Reliability (real NHAMCS)")
    ax[0].legend()

    d = EXT["rte"].copy()
    d["pred"] = EXT["pred"]
    d = d[d["hr"] == 1]
    axis = next((c for c in ["insurance", "sex", "race"] if c in d.columns), None)
    if axis is not None:
        g = d.groupby(axis)["pred"].apply(lambda s: (s > 2).mean()).sort_values(ascending=False)
        ax[1].bar(g.index.astype(str), g.values, color="#B279A2")
        ax[1].set_title(f"Undertriage (true ESI 1-2) by {axis}, real data")
        ax[1].set_ylabel("undertriage rate")
        ax[1].tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(os.path.join(ASSETS, "external_calibration_fairness.png"), dpi=90)
    plt.show()

## 24. Export the demo model

Fits the multiclass ESI text model on all training rows and saves it with the escalation gate
settings and the conformal threshold, so a small demo app can load it.

In [ ]:
import joblib

esi_demo = Pipeline([
    ("tfidf", FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2))])),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
])
esi_demo.fit(train["cc_clean"], train[TARGET])
_dcls = [int(c) for c in esi_demo.classes_]
_dhi = [i for i, c in enumerate(_dcls) if c <= 2]

_oof = cross_val_predict(clone(esi_demo), train["cc_clean"], train[TARGET],
                         cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")
demo_threshold = pick_threshold((train[TARGET] <= 2).astype(int).to_numpy(), _oof[:, _dhi].sum(1))

_full_vocab = sorted(CountVectorizer(token_pattern=r"[a-z]+").fit(train["cc_clean"]).vocabulary_.keys())
_conf_thr = float(np.quantile(_oof.max(1), 0.05))
_n = len(train)
_dtrue = np.array([_dcls.index(int(c)) for c in train[TARGET]])
_alpha = 0.10
_conformal_qhat = float(np.quantile(1 - _oof[np.arange(_n), _dtrue],
                                    min(np.ceil((_n + 1) * (1 - _alpha)) / _n, 1.0), method="higher"))

joblib.dump({"esi_model": esi_demo, "esi_classes": _dcls, "threshold": float(demo_threshold),
             "abbrev": ABBREV, "full_vocab": _full_vocab, "conf_thr": _conf_thr, "oov_thr": 0.5,
             "conformal_qhat": _conformal_qhat, "conformal_alpha": _alpha},
            "triage_text_model.joblib")
print(f"saved triage_text_model.joblib: threshold={demo_threshold:.3f}, conf_thr={_conf_thr:.3f}, "
      f"qhat={_conformal_qhat:.3f}, vocab={len(_full_vocab)}")

## 25. Clinical use

This is a decision-support flag. It does not replace the triage nurse. It would run at intake
and raise a second-look alert on presentations that look high-risk, tuned so that misses are
rare, at the cost of some extra alerts. The TF-IDF model runs in well under a millisecond on a
CPU, which makes it the realistic option at the bedside. The transformer is there as a
comparison.

## 26. Limitations

- The competition data is synthetic. Its distributions are cleaner than real practice and the
  complaint text is partly templated, so the near-perfect scores on it say little about real
  performance. We make no claim of clinical validity.
- `news2_score` almost determines the label in this data. It is excluded from the main
  structured model and both versions are reported.
- The external validation is a single re-fit on a different real dataset. It is not a
  prospective study, and the trained weights were not transferred because the schemas differ.
- Subgroup estimates in the fairness audit rest on small counts from a 20% validation split.

## 27. Responsible use

- Fairness: section 16 measures undertriage by group instead of assuming the model is fair.
  Any group with a higher miss rate is a safety issue to raise with clinical staff.
- Oversight: the flag should support the triage nurse, never replace them. Too much trust in a
  high-sensitivity alert is its own risk.
- Transparency: the deployable model is linear, so every keyword weight can be inspected by a
  clinician.

## 28. Conclusion

On the synthetic data every model separates ESI 1-2 from the rest almost perfectly, so the
choices that matter are the threshold, the vitals override, the escalation gate and the
fairness audit. On real NHAMCS visits the same text model is far weaker, which is the number
to keep in mind for any real deployment. The useful parts of this work are the safety layers
around the model and the checks on the data, more than the model's score on synthetic records.